# Case Study: Analysis of Visual Search Strategies (Where's Waldo)
**Module:** Foundations of Academic Research  
**Topic:** Investigation of the Reading-Pattern Hypothesis (Top-Down, Left-to-Right) using Eye-Tracking Data & Machine Learning Classifier

---

### Table of Contents
1. **Setup & Data Import**: Loading and cleaning raw data (`Data export`) and aggregated data (`Metrics`).
2. **Exploratory Data Analysis & Visualization**: Statistical analysis of participant metrics using Seaborn.
3. **Hypothesis Testing (The Reading Strategy)**: Temporal analysis of $X$- and $Y$-coordinates to validate systematic patterns (sawtooth & top-down).
4. **Feature Engineering & Machine Learning**: Classification of search success/behavior based on extracted features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from PIL import Image
import os

# Matplotlib & Seaborn styling for scientific papers
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully!")

## 1. Data Loading & Preprocessing
We load the complete **Data Export** (for coordinate time series) and the **Metrics Export** (for aggregated participant metrics).

In [ ]:
# 1. Load data (tab-separated)
df_data = pd.read_csv('data/EHBR Data export.tsv', sep='\t', low_memory=False)
df_metrics = pd.read_csv('data/EHBR Metrics.tsv', sep='\t')

# Clean column names
df_data.columns = df_data.columns.str.strip()
df_metrics.columns = df_metrics.columns.str.strip()

# Remove test recordings
df_data = df_data[~df_data['Participant name'].isin(['Test', 'Test2'])]
df_metrics = df_metrics[~df_metrics['Participant'].isin(['Test', 'Test2'])]

print(f"Data Export loaded: {df_data.shape[0]} rows, {df_data.shape[1]} columns.")
print(f"Metrics Export loaded: {df_metrics.shape[0]} rows, {df_metrics.shape[1]} columns.")

## 2. Exploratory Data Analysis (Metrics)
Here we analyze the distributions of fixations and saccades across the different search images (`TOI`).

In [ ]:
# Filter for primary search images (excluding 'Entire Recording')
walter_metrics = df_metrics[df_metrics['TOI'].isin(['Walter1', 'Walter2', 'Walter3'])].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot: Number of fixations per image
sns.boxplot(data=walter_metrics, x='TOI', y='Number_of_whole_fixations', hue='TOI', legend=False, ax=ax1, palette='Set2')
ax1.set_title('Number of Fixations per Search Image')
ax1.set_xlabel('Search Image (Time of Interest)')
ax1.set_ylabel('Fixations (Count)')

# Barplot: Average interval duration (search time)
sns.barplot(data=walter_metrics, x='TOI', y='Duration_of_interval', hue='TOI', legend=False, ax=ax2, palette='Set2', errorbar=None)
ax2.set_title('Average Search Time per Image (ms)')
ax2.set_xlabel('Search Image (Time of Interest)')
ax2.set_ylabel('Duration in Milliseconds')

plt.tight_layout()
plt.show()

## 3. Testing the Research Hypothesis: Do Humans Search Like They Read?
**Hypothesis:** Over time, the gaze moves continuously from top to bottom ($Y$-coordinate increases) and shows a cyclic sawtooth pattern horizontally ($X$-coordinate rises line by line from left to right and abruptly resets).

In [ ]:
# Select the first valid test recording from the Data Export
available_participants = df_data['Participant name'].dropna().unique()
print("Available participants in dataset:", available_participants)

# Select the first real participant and a Waldo image
chosen_participant = available_participants[0]
chosen_media = 'Walter1.jpg'

# Filter and remove missing gaze points
df_hyp = df_data[(df_data['Participant name'] == chosen_participant) & 
                 (df_data['Presented Media name'] == chosen_media)].copy()

df_hyp = df_hyp.dropna(subset=['Gaze point X', 'Gaze point Y'])

# Calculate time axis in seconds from stimulus onset
df_hyp['Time_Sec'] = (df_hyp['Computer timestamp'] - df_hyp['Computer timestamp'].min()) / 1_000_000

print(f"Analysis for {chosen_participant} on {chosen_media} with {len(df_hyp)} valid gaze points.")

In [ ]:
# Visualization of X and Y gaze paths over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# 1. Horizontal gaze trajectory (X-axis)
sns.lineplot(data=df_hyp, x='Time_Sec', y='Gaze point X', ax=ax1, color='royalblue', alpha=0.8)
ax1.set_title(f'Horizontal Gaze Trajectory (X) over Time - Participant: {chosen_participant}')
ax1.set_ylabel('Pixel (Left -> Right)')
ax1.grid(True)

# 2. Vertical gaze trajectory (Y-axis)
sns.lineplot(data=df_hyp, x='Time_Sec', y='Gaze point Y', ax=ax2, color='forestgreen', alpha=0.8)
ax2.set_title(f'Vertical Gaze Trajectory (Y) over Time - Participant: {chosen_participant}')
ax2.set_xlabel('Time (Seconds)')
ax2.set_ylabel('Pixel (Top -> Bottom)')
ax2.invert_yaxis()  # Important: pixel 0 is at the top of the screen!
ax2.grid(True)

plt.tight_layout()
plt.show()

# Statistical validation using Spearman rank correlation
rho_x, p_x = spearmanr(df_hyp['Time_Sec'], df_hyp['Gaze point X'])
rho_y, p_y = spearmanr(df_hyp['Time_Sec'], df_hyp['Gaze point Y'])

print(f"--- Statistical Results for {chosen_participant} ---")
print(f"Spearman correlation (Time vs. X): {rho_x:.3f} (p-value: {p_x:.4f})")
print(f"Spearman correlation (Time vs. Y): {rho_y:.3f} (p-value: {p_y:.4f}) -> A positive value supports the Reading Hypothesis (Top-Down)!")

In [ ]:
# 1. Prepare data (using the already filtered 'df_hyp' from step 3)
if 'df_hyp' in locals() and not df_hyp.empty:

    # Ensure no copy warnings
    df_phase = df_hyp.copy()

    # 2. Divide search time into 3 equal segments (tertiles)
    # qcut ensures each phase contains an equal number of data points
    df_phase['Phase'] = pd.qcut(df_phase['Time_Sec'], q=3, labels=['Beginning', 'Middle', 'End'])

    # 3. Create plot
    plt.figure(figsize=(10, 6))

    # Boxplot for Y-coordinate per phase
    sns.boxplot(
        data=df_phase,
        x='Phase',
        y='Gaze point Y',
        palette='YlGnBu',
        hue='Phase',
        legend=False
    )

    # 4. Axis inversion (critical for eye-tracking data!)
    plt.gca().invert_yaxis()

    # Labels
    plt.title(f'Gaze Height (Y-Coordinate) over Time\nParticipant: {chosen_participant} | Stimulus: {chosen_media}', fontsize=14, pad=15)
    plt.xlabel('Search Phase (Chronological Division into 3 Segments)', fontsize=12, labelpad=10)
    plt.ylabel('Vertical Position on Screen (Pixel)', fontsize=12)

    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    # 5. Descriptive statistics for the written report
    print("--- Descriptive Statistics of Gaze Height (Y) per Phase ---")
    print(df_phase.groupby('Phase', observed=False)['Gaze point Y'].mean().to_string())

else:
    print("Error: Please run the cell from step 3 first to define 'df_hyp'.")

## 3b. Spearman Correlation for All Participants & Images
We calculate the correlation between time and Y-coordinate (top-down) as well as time and X-coordinate (left-right) for all combinations of participant and stimulus.

In [ ]:
# Compute Spearman correlation for all participants × all stimuli
stimuli_files = {'Walter1': 'Walter1.jpg', 'Walter2': 'Walter2.jpg', 'Walter3': 'Walter3.jpg'}
corr_records = []

for pid in sorted(df_data['Participant name'].dropna().unique()):
    for stim_label, stim_file in stimuli_files.items():
        subset = df_data[
            (df_data['Participant name'] == pid) &
            (df_data['Presented Media name'] == stim_file)
        ].dropna(subset=['Gaze point X', 'Gaze point Y']).copy()

        if len(subset) < 10:
            continue

        t = (subset['Computer timestamp'] - subset['Computer timestamp'].min()) / 1_000_000
        rho_y, p_y = spearmanr(t, subset['Gaze point Y'])
        rho_x, p_x = spearmanr(t, subset['Gaze point X'])

        corr_records.append({
            'Participant': pid, 'Stimulus': stim_label,
            'rho_Y (Top-Down)': round(rho_y, 3), 'p_Y': round(p_y, 4),
            'rho_X (Left-Right)': round(rho_x, 3), 'p_X': round(p_x, 4),
            'sig_Y': '***' if p_y < 0.001 else ('**' if p_y < 0.01 else ('*' if p_y < 0.05 else '–')),
            'sig_X': '***' if p_x < 0.001 else ('**' if p_x < 0.01 else ('*' if p_x < 0.05 else '–')),
        })

df_corr = pd.DataFrame(corr_records)
print(df_corr[['Participant','Stimulus','rho_Y (Top-Down)','sig_Y','rho_X (Left-Right)','sig_X']].to_string(index=False))
print()
print("Means:")
print(f"  rho_Y (Top-Down):  {df_corr['rho_Y (Top-Down)'].mean():.3f}")
print(f"  rho_X (Left-Right): {df_corr['rho_X (Left-Right)'].mean():.3f}")

In [ ]:
# Heatmap visualization: rho_Y and rho_X for all participants × stimuli
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes,
    ['rho_Y (Top-Down)', 'rho_X (Left-Right)'],
    ['Top-Down Tendency (rho_Y)\n+1 = Gaze moves consistently downward',
     'Left-Right Tendency (rho_X)\n+1 = Gaze moves consistently rightward']):

    pivot = df_corr.pivot(index='Participant', columns='Stimulus', values=col)
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
                vmin=-1, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

## 4. Heatmap Analysis of Gaze Data
Pre-computed heatmap images are available for each participant. We analyze these programmatically to answer two questions:
1. **Concentration**: How focused was the search? (Share of actively viewed image area)
2. **Vertical distribution**: Is activity concentrated at the top or bottom? (Supports the Top-Down Hypothesis)

In [ ]:
import glob as _glob

def find_heatmap(pid, stim):
    """Finds the heatmap file regardless of spacing variations in the filename."""
    matches = _glob.glob(f'data/{pid}/{stim}*Heat*.png')
    return matches[0] if matches else None

def analyse_heatmap(path):
    img = np.array(Image.open(path))
    r, g, b = img[:,:,0].astype(int), img[:,:,1].astype(int), img[:,:,2].astype(int)
    warm = (r - g > 40) & (r - b > 40)
    coverage = warm.mean()
    h = warm.shape[0]
    top_act    = warm[:h//3].mean()
    mid_act    = warm[h//3:2*h//3].mean()
    bottom_act = warm[2*h//3:].mean()
    row_weights = warm.sum(axis=1)
    center_y = np.average(np.arange(h), weights=row_weights) / h if row_weights.sum() > 0 else np.nan
    return coverage, center_y, top_act, mid_act, bottom_act

participants = sorted([d for d in os.listdir('data') if d.startswith('ID')])
stimuli = ['Walter1', 'Walter2', 'Walter3']

records = []
for pid in participants:
    for stim in stimuli:
        path = find_heatmap(pid, stim)
        if not path:
            print(f"WARNING: No heatmap found for {pid} / {stim}")
            continue
        cov, cy, top, mid, bot = analyse_heatmap(path)
        records.append({
            'Participant': pid, 'Stimulus': stim,
            'Area_Coverage_%': round(cov * 100, 2),
            'Vertical_Center': round(cy, 3),
            'Activity_Top': round(top * 100, 2),
            'Activity_Middle': round(mid * 100, 2),
            'Activity_Bottom': round(bot * 100, 2),
        })

df_heat = pd.DataFrame(records)
print(f"{len(df_heat)} heatmaps analyzed.\n")
print(df_heat.to_string(index=False))

In [ ]:
# 4a. Overview plot: Area coverage & vertical center per participant and stimulus
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=df_heat, x='Participant', y='Area_Coverage_%', hue='Stimulus',
            palette='Set2', ax=axes[0])
axes[0].set_title('Actively Viewed Image Area per Participant (%)\n(less = more focused search)')
axes[0].set_ylabel('Area Coverage (%)')
axes[0].set_xlabel('')

sns.barplot(data=df_heat, x='Participant', y='Vertical_Center', hue='Stimulus',
            palette='Set2', ax=axes[1])
axes[1].set_title('Vertical Center of Attention\n(0 = top, 1 = bottom)')
axes[1].set_ylabel('Normalized Center (0–1)')
axes[1].set_xlabel('')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.7, label='Image center')
axes[1].legend()

plt.tight_layout()
plt.show()

# 4b. Third distribution: How much activity is at the top vs. bottom?
df_thirds = df_heat.melt(
    id_vars=['Participant', 'Stimulus'],
    value_vars=['Activity_Top', 'Activity_Middle', 'Activity_Bottom'],
    var_name='Image_Third', value_name='Activity_%'
)
df_thirds['Image_Third'] = df_thirds['Image_Third'].str.replace('Activity_', '')

plt.figure(figsize=(10, 5))
sns.barplot(data=df_thirds, x='Image_Third', y='Activity_%', hue='Stimulus',
            palette='Set2', order=['Top', 'Middle', 'Bottom'])
plt.title('Average Heatmap Activity per Image Third\n(Evidence for Top-Down Search Pattern)')
plt.ylabel('Activity (%)')
plt.xlabel('Image Third (Vertical)')
plt.tight_layout()
plt.show()

In [ ]:
# 4c. Heatmap overview: All participants for Walter1 side by side
stim_to_show = 'Walter1'
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, pid in enumerate(participants):
    path = find_heatmap(pid, stim_to_show)
    if path:
        axes[i].imshow(Image.open(path))
        axes[i].set_title(pid, fontsize=12)
    axes[i].axis('off')

for j in range(len(participants), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f'Heatmap Overview: {stim_to_show} – All Participants', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()